In [1]:
import io
import os
import SimpleITK as sitk
import pandas as pd
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure, color
import skimage.morphology as morphology
from skimage.segmentation import mark_boundaries as mark_boundaries
from skimage.segmentation import slic as slic
import shutil
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
import nibabel as nib
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import copy
import shutil

In [4]:
def preprocess_mri(image_path, mask_path=None, output_image_path=None, output_mask_path=None):
    """
    Perform three-step preprocessing on MRI data:
    1. N4 Bias Field Correction (with optional mask)
    2. Resampling to isotropic voxels (1 mm)
    3. Histogram standardization

    Args:
        image_path (str): Path to the input MRI image (NIfTI format).
        mask_path (str, optional): Path to the mask image (NIfTI format) for bias field correction.
        output_image_path (str, optional): Path to save the preprocessed image.
        output_mask_path (str, optional): Path to save the resampled mask. Required if mask_path is provided.

    Returns:
        sitk.Image: Preprocessed MRI image.
    """
    # Step 1: N4 Bias Field Correction
#     print("Step 1: N4 Bias Field Correction")
    image = sitk.ReadImage(image_path, sitk.sitkFloat32)

    corrector = sitk.N4BiasFieldCorrectionImageFilter()

    mask = sitk.ReadImage(mask_path, sitk.sitkUInt8)
    corrected_image = corrector.Execute(image, mask)
   
    #     corrected_image = corrector.Execute(image)

    # Step 2: Resampling to isotropic voxels (1 mm)
#     print("Step 2: Resampling to isotropic voxels")
    original_spacing = corrected_image.GetSpacing()
    original_size = corrected_image.GetSize()
    new_spacing = (1.0, 1.0, 1.0)
    new_size = [
        int(np.round(original_size[i] * (original_spacing[i] / new_spacing[i])))
        for i in range(3)
    ]
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(corrected_image.GetDirection())
    resampler.SetOutputOrigin(corrected_image.GetOrigin())
    resampler.SetInterpolator(sitk.sitkBSpline)
    resampled_image = resampler.Execute(corrected_image)

    if mask_path:
        # Resample the mask using nearest neighbor interpolation
        print("Resampling mask to isotropic voxels")
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
        resampled_mask = resampler.Execute(mask)

    # Step 3: Histogram standardization
#     print("Step 3: Histogram standardization")
    array = sitk.GetArrayFromImage(resampled_image)
    standardized_array = (array - np.mean(array)) / np.std(array)  # Z-score normalization
    standardized_image = sitk.GetImageFromArray(standardized_array)
    standardized_image.CopyInformation(resampled_image)

    # Save the preprocessed image and mask if output paths are provided
    if output_image_path:
        print(f"Saving preprocessed image to {output_image_path}")
        sitk.WriteImage(standardized_image, output_image_path)

    if mask_path and output_mask_path:
        print(f"Saving resampled mask to {output_mask_path}")
        sitk.WriteImage(resampled_mask, output_mask_path)

    return standardized_image, resampled_mask if mask_path else standardized_image


In [ ]:
# Example usage
if __name__ == "__main__":
    
    img_dir = r"/DWI"
    roi_dir = r"/DWI_ROI"
    output_img_dir = r"/preprocess/DWI_IMG"
    output_roi_dir = r"/preprocess/DWI_ROI"
    
    
    plist = os.listdir(img_dir )
    plist.sort()
    try:
        plist.remove('.DS_Store')
    except:
        pass

    
    for i in plist:
        
    
        image_path = img_dir + '/' + i
#         mask_path = roi_dir + '/' + i
        
        mask_path = roi_dir + '/' + i + '.gz'
        
        output_image_path = output_img_dir + '/' + i
        output_mask_path = output_roi_dir + '/' + i
        print(image_path)

        preprocessed_image, preprocessed_mask = preprocess_mri(
            image_path, mask_path, output_image_path, output_mask_path)

In [ ]:
def check_dataset(list_volume, list_mask):
    if len(list_volume) != len(list_mask):
        print(len(list_volume))
        print(len(list_mask))
        raise ValueError('There exists a mismatch between two datasets.')

def mac_os_listdir(path):
    files = os.listdir(path)
    if '.DS_Store' in files:
        files.remove('.DS_Store')
        files.sort()
    return files

def get_file_num(path):
    files_num = []
    files = os.listdir(path)
    for file in files:
        if file !='.DS_Store':
            # print(file)
            num = ""
            for c in file:
                if c.isdigit():
                    num = num + c
            # print("Extracted numbers from the list : " + num)
            files_num.append(num)
    return files_num

In [ ]:
def supervoxel(volume_list, mask_list,  supervoxel_path):
    print('Start super voxel processing ...')
    if not os.path.exists(supervoxel_path):
        os.makedirs(supervoxel_path)

  

    for volume_path, mask_path in zip(volume_list, mask_list):
        
        IMG = sitk.ReadImage(volume_path)
        img = sitk.GetArrayFromImage(IMG)
        mask = sitk.GetArrayFromImage(sitk.ReadImage(mask_path))
        if np.sum(mask)>0:
            img_normalized = cv2.normalize(img, None, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)


            # 对掩膜 (mask) 进行形态学开运算，用来去除小物体或噪声
            tmp_mask = morphology.opening(mask, morphology.ball(1))

            if np.any(tmp_mask):

                # 计算 ROI（掩膜）中的体素个数
                roi_voxel_count = np.sum(tmp_mask > 0)  # 统计掩膜中非零体素的数量
            else:
                # 计算 ROI（掩膜）中的体素个数            
                roi_voxel_count = np.sum(mask > 0)  # 统计掩膜中非零体素的数量

            # 根据体素个数动态设置 n_segment 参数
            if roi_voxel_count < 1000:
                n_segment = 30
            elif 1000 <= roi_voxel_count < 10000:
                n_segment = 50
            elif 10000 <= roi_voxel_count < 100000:
                n_segment = 60
            else:
                n_segment = 100

            # 检查形态学处理后的掩膜是否有效（是否存在非零值）
            if np.any(tmp_mask):  # 如果 tmp_mask 中存在有效区域
                try:
                    # 使用 SLIC 算法进行超像素分割，仅在有效掩膜范围内分割
                    slic_mask = slic(
                        img_normalized,    # 输入图像，需为归一化后的版本
                        n_segment,         # 动态设置的目标超像素数量
                        compactness=10,    # 超像素的紧凑性，越大越接近立方体
                        mask=tmp_mask,     # 只在 tmp_mask 范围内进行分割
                        start_label=1,     # 超像素标签从 1 开始
                        channel_axis=None  # 指定图像通道的轴（灰度图时为 None）
                    )
                except Exception:  # 捕捉 SLIC 分割中的任何异常
                    print('something wrong, SV computation failed !!!')  # 输出错误信息
            else:  # 如果 tmp_mask 无效（全为 0）
                tmp_mask = mask  # 使用原始掩膜
                try:
                    # 使用原始掩膜重新进行 SLIC 超像素分割
                    slic_mask = slic(
                        img_normalized,
                        n_segment,         # 动态设置的目标超像素数量
                        compactness=10,
                        mask=tmp_mask,
                        start_label=1,
                        channel_axis=None
                    )
                except Exception:  # 捕捉 SLIC 分割中的任何异常
                    print('something wrong, SV computation failed !!!')  # 输出错误信息

            slic_mask_itk = sitk.GetImageFromArray(np.uint16(slic_mask))
            slic_mask_itk.CopyInformation(IMG)




            # save slic_mask
            output_name = os.path.join(supervoxel_path, mask_path.split("/")[-1].split('.')[0] + '_SVmask.nii.gz')
            sitk.WriteImage(slic_mask_itk, output_name)
            print('{} is Done'.format(output_name.split('/')[-1]))



In [ ]:


import os
##数据和roi路径
data_dir = r'/preprocess/DWI_IMG'
roi_dir = r"/DWI_ROI"
#组学特征提取yaml文件
params = r"/exampleMR_1mm.yaml"
#结果保存临时文件夹
dirpath = r"/preprocess/habitat_DWI"
#超像素分割数目
n_segment = 100
#文件夹创建
tmp_volume = os.path.join(dirpath,'tmp_dir','ImageDir')
tmp_volume = tmp_volume.replace(os.path.sep, '/')

tmp_mask = os.path.join(dirpath,'tmp_dir','MaskDir')
tmp_mask = tmp_mask.replace(os.path.sep, '/')
out_dir = os.path.join(dirpath,'output')
out_dir = out_dir.replace(os.path.sep, '/')
SVS_dir = os.path.join(out_dir,'SV','SVSplit')
SVS_dir = SVS_dir.replace(os.path.sep, '/')
SVSmask_dir = os.path.join(out_dir,'SV','SVSplit','SVmask')
SVSmask_dir = SVSmask_dir.replace(os.path.sep, '/')
SVSvolume_dir = os.path.join(out_dir,'SV','SVSplit','SVImage')
SVSvolume_dir = SVSvolume_dir.replace(os.path.sep, '/')
SVwhole_dir = os.path.join(out_dir,'SV','SVWhole')
SVwhole_dir = SVwhole_dir.replace(os.path.sep, '/')
casetable_dir = os.path.join(out_dir,'CaseTable')
casetable_dir = casetable_dir.replace(os.path.sep, '/')
if os.path.isdir(tmp_volume):
    shutil.rmtree(tmp_volume)

if os.path.isdir(tmp_mask):
    shutil.rmtree(tmp_mask)
dir_list = [out_dir,tmp_volume,tmp_mask,SVS_dir,SVSmask_dir,SVSvolume_dir,SVwhole_dir,casetable_dir]
for dir in dir_list:
    if not os.path.exists(dir):
        os.makedirs(dir)


# # 获取文件路径列表
image_list = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f !='.DS_Store'])

mask_list = sorted([os.path.join(roi_dir, f) for f in os.listdir(roi_dir) if f !='.DS_Store'])


image_list.sort()
mask_list.sort()

check_dataset(image_list, mask_list)


In [ ]:
# supervoxel calculation by slic, 

supervoxel(image_list, mask_list, SVwhole_dir)
print('Supervoxel Done')

In [ ]:
#读取图像和超像素文件
sv_list = os.listdir(SVwhole_dir)
sv_list.sort()
sv_list = [os.path.join(SVwhole_dir, file) for file in sv_list]
sv_list = [path.replace(os.path.sep, '/') for path in sv_list] 
image_list_new1 = copy.copy(image_list)

    
check_dataset(image_list_new1, sv_list)
print('Dataset checked!')

In [ ]:


#以下两个函数用于超像素分割后把每个超像素的ROI单独保存
def process_sv_split(image_i, mask_j, out_dir):
    print(f"Processing image: {image_i}, mask: {mask_j}")

    if not os.path.isfile(image_i):
        print(f"Error: {image_i} is not a valid file.")
        return
    if not os.path.isfile(mask_j):
        print(f"Error: {mask_j} is not a valid file.")
        return

    mask_array = sitk.GetArrayFromImage(sitk.ReadImage(mask_j))
    _, num = measure.label(mask_array, connectivity=3, return_num=True)

    for i in range(1, num + 1):
        section_volume = sitk.GetArrayFromImage(sitk.ReadImage(image_i))
        section_mask = copy.copy(mask_array)

        section_mask[section_mask != i] = 0
        section_mask[section_mask == i] = 1

        svsplit_path = os.path.join(out_dir, 'SVmask',  mask_j.split('/')[-1].split('.')[-3][:-7])

        if not os.path.exists(svsplit_path):
            os.makedirs(svsplit_path)

        mask_i = sitk.GetImageFromArray(section_mask)
        mask_i.CopyInformation(sitk.ReadImage(mask_j))

        sitk.WriteImage(mask_i, os.path.join(svsplit_path, mask_j.split('/')[-1].split('.')[-3]) + f'_{i}.nii.gz')




def SV_split(volume_list, sv_list, out_dir):
    print('Start SV split processing ...')

    with ThreadPoolExecutor(max_workers=10) as executor:  # 设置并行线程数为 10
        futures = []
        for image_i, mask_j in zip(volume_list, sv_list):
            futures.append(executor.submit(process_sv_split, image_i, mask_j, out_dir))

        for future in futures:
            future.result()  # 捕获可能的异常

print('SV split done')

# 并行执行 SV split 处理
#########################
SV_split(image_list_new1, sv_list, SVS_dir)
print('SV split done')

In [ ]:
## for radiomics - prepare casetable
################################
image_files = image_list_new1
mask_files = mac_os_listdir(SVSmask_dir)

image_files.sort()
mask_files.sort()



check_dataset(image_files, mask_files)

for i in range(0,len(image_files)):
    print(i)
    #img_id = image_files[i].split('/')[-1].split('.')[-3]
    img_id = image_files[i].split('/')[-1].split('.')[0]
    dirsv = os.path.join(SVSmask_dir,mask_files[i])
    dirsv = dirsv.replace(os.path.sep, '/')
    SV_files = mac_os_listdir(dirsv)
    SV_files.sort()
    CaseTable = pd.DataFrame()
    for j in range(0,len(SV_files)):
        ptid = pd.Series({'ID':SV_files[j].split('.')[-3]})
        Image_path = pd.Series({'Image':image_files[i]})
        dirtmp = os.path.join(SVSmask_dir,mask_files[i],SV_files[j])
        dirtmp = dirtmp.replace(os.path.sep, '/')
        Mask_path = pd.Series({'Mask':dirtmp})

        case_id_img = pd.concat([ptid, Image_path])
        case_id_img_mask = pd.concat([case_id_img, Mask_path])
        CaseTable = pd.concat([CaseTable,case_id_img_mask],axis = 1)

    # generate case table for radiomics calculation, including ID, image and mask path
    dirtmp = os.path.join(casetable_dir, img_id +'_CaseTable.csv')
    dirtmp = dirtmp.replace(os.path.sep, '/')
    #CaseTable.T.to_csv(dirtmp),index=False)
    CaseTable.T.to_csv(dirtmp,index=False)
alltable = pd.DataFrame({})
for i in mac_os_listdir(casetable_dir):
    tmppath = os.path.join(casetable_dir,i)
    tmppath = tmppath.replace(os.path.sep, '/')
    tmpcsv = pd.read_csv(tmppath)
    alltable = pd.concat([alltable,tmpcsv])
# dirtmp = os.path.join(dirpath,'output','alltable.csv')
# dirtmp = dirtmp.replace(os.path.sep, '/')
# alltable.to_csv(dirtmp,index=False)

alltable.to_csv(os.path.join(dirpath,'output','alltable.csv'),index=False)



#组学特征提取
def extract_features(entry, flists, extractor):
    imageFilepath = flists[entry]['Image']
    maskFilepath = flists[entry]['Mask']
    label = flists[entry].get('Label', None)

    if str(label).isdigit():
        label = int(label)
    else:
        label = None

    featureVector = flists[entry]
    featureVector['Image'] = os.path.basename(imageFilepath)
    featureVector['Mask'] = os.path.basename(maskFilepath)

    try:
        result = pandas.Series(extractor.execute(imageFilepath, maskFilepath, label))
        # Use pd.concat instead of append
        featureVector = pd.concat([featureVector, result])
    except Exception as e:
        logger.error(f'Feature extraction failed for {entry}: {e}', exc_info=True)
        return None

    return featureVector




#!/usr/bin/env python

from __future__ import print_function

import logging
import os

import pandas
import SimpleITK as sitk

import radiomics
from radiomics import featureextractor


# def main():



inputCSV = os.path.join(dirpath,'output','alltable.csv')

outputFilepath = os.path.join(dirpath, 'radiomics_featuretable.csv')

progress_filename = os.path.join('./pyrad_log.txt')

# Configure logging
rLogger = logging.getLogger('radiomics')


handler = logging.FileHandler(filename=progress_filename, mode='w')
handler.setFormatter(logging.Formatter('%(levelname)s:%(name)s: %(message)s'))
rLogger.addHandler(handler)
logger = rLogger.getChild('batch')
radiomics.setVerbosity(logging.INFO)
logger.info('pyradiomics version: %s', radiomics.__version__)
logger.info('Loading CSV')
try:
    flists = pandas.read_csv(inputCSV).T

    
except Exception:
    logger.error('CSV READ FAILED', exc_info=True)
    exit(-1)

logger.info('Loading Done')
logger.info('Patients: %d', len(flists.columns))

if os.path.isfile(params):
    extractor = featureextractor.RadiomicsFeatureExtractor(params)
else:  
    settings = {}
    settings['binWidth'] = 25
    settings['resampledPixelSpacing'] = None  # [3,3,3]
    settings['interpolator'] = sitk.sitkBSpline
    settings['enableCExtensions'] = True

    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)

logger.info('Enabled input images types: %s', extractor.enabledImagetypes)
logger.info('Enabled features: %s', extractor.enabledFeatures)
logger.info('Current settings: %s', extractor.settings)

results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    future_to_entry = {executor.submit(extract_features, entry, flists, extractor): entry for entry in
                       flists.columns}

    for future in future_to_entry:
        entry = future_to_entry[future]
        try:
            result = future.result()
            if result is not None:
                results.append(result)
        except Exception as e:
            logger.error(f'Processing entry {entry} failed: {e}', exc_info=True)

# 将结果转化为 DataFrame 并保存
if results:
    results_df = pandas.DataFrame(results)
    logger.info('Extraction complete, writing CSV')
    results_df.to_csv(outputFilepath, index=False, na_rep='NaN')
    logger.info('CSV writing complete')
else:
    logger.info('No valid results to write.')


In [ ]:
cols = [i for i in results_df.columns.tolist() if not i.startswith('diagnostics')]
results_df = results_df[cols]
results_df.to_csv(r"./hatfeas.csv")